In [0]:
%sql
CREATE OR REPLACE TABLE data_warehouse_factory.gold.fct_daily_employee_utilization AS
WITH 
-- 1. Wyciągnięcie zakresu dat z planu produkcji i pobranie z dim_date tylko dni roboczych
working_days AS (
  SELECT 
    d.date AS production_date
  FROM data_warehouse_factory.gold.dim_date d
  WHERE d.is_working_day = TRUE
    AND d.date BETWEEN (SELECT MIN(plan_date) FROM data_warehouse_factory.silver.silver_production_plan)
                   AND (SELECT MAX(plan_date) FROM data_warehouse_factory.silver.silver_production_plan)
),

-- 2. Aktywni pracownicy w analizowanym okresie
active_employees AS (
  SELECT 
    employee_key, 
    employee_full_name
  FROM data_warehouse_factory.silver.silver_employees
  WHERE date_of_employment IS NOT NULL 
    AND (date_of_leaving IS NULL OR date_of_leaving >= CURRENT_DATE())
),

-- 3. Siatka Dostępności: Każdy pracownik x Dzień roboczy z dim_date (Norma 8.0h)
employee_days_capacity AS (
  SELECT 
    d.production_date,
    e.employee_key,
    e.employee_full_name,
    8.0 AS total_capacity_hours
  FROM working_days d
  CROSS JOIN active_employees e
),

-- 4. Wyliczenie godzin przepracowanych z Zaplanowanej Produkcji
planned_hours_worked AS (
  SELECT 
    start_date AS production_date,
    assigned_operator_key AS employee_key,
    ROUND(SUM(timestampdiff(MINUTE, interval_start, interval_end)) / 60.0, 2) AS planned_production_hours
  FROM data_warehouse_factory.gold.fct_final_operator_assignments
  WHERE assigned_operator_key IS NOT NULL
  GROUP BY start_date, assigned_operator_key
),

-- 5. Wyliczenie godzin z Dodatkowych Aktywności / Zdarzeń
additional_activities_hours AS (
  SELECT 
    production_date,
    employee_key,
    ROUND(SUM(duration_minutes) / 60.0, 2) AS additional_activity_hours
  FROM data_warehouse_factory.silver.silver_events_daily
  WHERE event_type NOT IN ('62-Absence', 'Absence', 'Przerwa')
  GROUP BY production_date, employee_key
),

-- 6. Złożenie bilansu i wyliczenie czasu niewykorzystanego (Unutilized)
calculated_utilization AS (
  SELECT 
    c.production_date,
    c.employee_key,
    c.employee_full_name,
    c.total_capacity_hours,
    
    COALESCE(p.planned_production_hours, 0.0) AS planned_production_hours,
    COALESCE(a.additional_activity_hours, 0.0) AS additional_activity_hours,
    
    -- Wyliczenie łącznego wykorzystanego czasu (maksymalnie 8h w standardowym bilansie)
    LEAST(
      c.total_capacity_hours, 
      COALESCE(p.planned_production_hours, 0.0) + COALESCE(a.additional_activity_hours, 0.0)
    ) AS total_utilized_hours,
    
    -- Wyliczenie pustego/nieobsadzonego czasu w dany dzień roboczy (8h - Wykorzystany czas)
    GREATEST(
      0.0, 
      c.total_capacity_hours - (COALESCE(p.planned_production_hours, 0.0) + COALESCE(a.additional_activity_hours, 0.0))
    ) AS unutilized_hours
  FROM employee_days_capacity c
  LEFT JOIN planned_hours_worked p 
    ON c.production_date = p.production_date AND c.employee_key = p.employee_key
  LEFT JOIN additional_activities_hours a 
    ON c.production_date = a.production_date AND c.employee_key = a.employee_key
)

SELECT 
  md5(concat_ws('||', cast(production_date as string), cast(employee_key as string))) AS utilization_key,
  CAST(date_format(production_date, 'yyyyMMdd') AS INT) AS date_key, -- Klucz obcy do dim_date
  production_date,
  employee_key,
  employee_full_name,
  total_capacity_hours,
  planned_production_hours,
  additional_activity_hours,
  total_utilized_hours,
  unutilized_hours,
  
  -- Statyczny wskaźnik dzienny (W Power BI zalecane użycie miary DAX)
  ROUND((total_utilized_hours / total_capacity_hours) * 100, 2) AS utilization_pct,
  
  CURRENT_TIMESTAMP() AS _created_at
FROM calculated_utilization;